# Kaggle: rebuild hybrid RAG index

Этот ноутбук собирает индекс **текущим кодом репозитория**, а не вручную.

Что делает:
- ставит зависимости для ingestion/retrieval;
- клонирует репозиторий;
- находит `records.jsonl` в `../input`;
- опционально подхватывает `manual_docs/`;
- скачивает embedding и reranker модели;
- запускает `build_hybrid_index(...)`;
- делает smoke test загрузки `full` retriever;
- упаковывает артефакты в `rag_index.tar.gz`.

Ожидания от Kaggle:
- runtime: Python + GPU желательно, но не обязательно;
- в `Add data` прикреплён датасет с `records.jsonl`;
- интернет включён, если модели надо качать с Hugging Face.


In [ ]:
# Cell 1 — config
from pathlib import Path

REPO_URL = 'https://github.com/chudinovAI/llm-speaker-core.git'
REPO_REF = 'main'

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'llm-speaker-core'
OUTPUT_ROOT = WORK_ROOT / 'rag_build'
DATA_ROOT = OUTPUT_ROOT / 'data'
MODEL_CACHE_ROOT = WORK_ROOT / 'hf-cache'
ARCHIVE_PATH = WORK_ROOT / 'rag_index.tar.gz'

RAW_RECORDS_OVERRIDE = None
MANUAL_DOCS_OVERRIDE = None
EMBEDDING_MODEL_OVERRIDE = None
RERANKER_MODEL_OVERRIDE = None

RAW_RECORDS_FILENAME = 'records.jsonl'
MANUAL_DOCS_DIRNAME = 'manual_docs'
EMBEDDING_MODEL = 'ai-sage/Giga-Embeddings-instruct'
RERANKER_MODEL = 'BAAI/bge-reranker-v2-m3'
SMOKE_TEST_QUERY = 'Какой режим работы отдела кадров ГУАП?'

for path in [OUTPUT_ROOT, DATA_ROOT, MODEL_CACHE_ROOT, DATA_ROOT / 'indexes/bm25', DATA_ROOT / 'indexes/faiss', DATA_ROOT / 'normalized']:
    path.mkdir(parents=True, exist_ok=True)

print('WORK_ROOT =', WORK_ROOT)
print('REPO_DIR =', REPO_DIR)
print('DATA_ROOT =', DATA_ROOT)


In [ ]:
# Cell 2 — install build dependencies
import subprocess
import sys

packages = [
    'bs4>=0.0.2',
    'lxml>=5.2.2',
    'faiss-cpu>=1.12.0',
    'huggingface-hub>=0.34.4',
    'numpy>=1.26.4',
    'pypdf>=6.4.2',
    'python-docx>=1.2.0',
    'flagembedding>=1.3.5',
    'sentence-transformers>=5',
    'transformers<5',
    'einops>=0.8',
    'html2text>=2025.4.15',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Dependencies installed')


In [ ]:
# Cell 3 — clone repo and add src to PYTHONPATH
import shutil
import subprocess
import sys

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
if REPO_REF and REPO_REF != 'main':
    subprocess.check_call(['git', 'checkout', REPO_REF], cwd=REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print('Repo ready:', REPO_DIR)
print('src added:', src_dir)


In [ ]:
# Cell 4 — resolve datasets and model paths
from huggingface_hub import snapshot_download


def pick_largest(paths):
    items = [Path(p) for p in paths if Path(p).exists()]
    if not items:
        return None
    items.sort(key=lambda p: p.stat().st_size, reverse=True)
    return items[0]


def find_records_jsonl():
    if RAW_RECORDS_OVERRIDE:
        path = Path(RAW_RECORDS_OVERRIDE)
        assert path.exists(), f'RAW_RECORDS_OVERRIDE does not exist: {path}'
        return path
    candidates = list(KAGGLE_INPUT_ROOT.rglob(RAW_RECORDS_FILENAME))
    path = pick_largest(candidates)
    assert path is not None, 'records.jsonl not found in /kaggle/input'
    return path


def find_manual_docs_dir():
    if MANUAL_DOCS_OVERRIDE:
        path = Path(MANUAL_DOCS_OVERRIDE)
        assert path.exists(), f'MANUAL_DOCS_OVERRIDE does not exist: {path}'
        return path
    candidates = [p for p in KAGGLE_INPUT_ROOT.rglob(MANUAL_DOCS_DIRNAME) if p.is_dir()]
    return sorted(candidates)[0] if candidates else None


def resolve_model_path(model_name: str, override: str | None, target_dir: Path) -> str:
    if override:
        path = Path(override)
        assert path.exists(), f'Model override does not exist: {path}'
        return str(path)
    target_dir.mkdir(parents=True, exist_ok=True)
    path = snapshot_download(repo_id=model_name, local_dir=str(target_dir))
    return str(path)


RAW_RECORDS = find_records_jsonl()
MANUAL_DOCS_DIR = find_manual_docs_dir()
EMBEDDING_MODEL_PATH = resolve_model_path(EMBEDDING_MODEL, EMBEDDING_MODEL_OVERRIDE, MODEL_CACHE_ROOT / 'embedding')
RERANKER_MODEL_PATH = resolve_model_path(RERANKER_MODEL, RERANKER_MODEL_OVERRIDE, MODEL_CACHE_ROOT / 'reranker')

print('RAW_RECORDS =', RAW_RECORDS)
print('RAW_RECORDS size MB =', round(RAW_RECORDS.stat().st_size / 1024 / 1024, 2))
print('MANUAL_DOCS_DIR =', MANUAL_DOCS_DIR)
print('EMBEDDING_MODEL_PATH =', EMBEDDING_MODEL_PATH)
print('RERANKER_MODEL_PATH =', RERANKER_MODEL_PATH)


In [ ]:
# Cell 5 — build index with repo pipeline
import json
from llm_speaker_core.retrieval.build import build_hybrid_index

report = build_hybrid_index(
    raw_records=RAW_RECORDS,
    documents_out=DATA_ROOT / 'normalized/documents.jsonl',
    chunks_out=DATA_ROOT / 'normalized/chunks.jsonl',
    manifest_out=DATA_ROOT / 'index_manifest.json',
    lexical_out=DATA_ROOT / 'indexes/bm25/index.json',
    dense_out=DATA_ROOT / 'indexes/faiss/index.json',
    embedding_model=EMBEDDING_MODEL,
    reranker_model=RERANKER_MODEL,
    manual_docs_dir=MANUAL_DOCS_DIR,
)

report_path = OUTPUT_ROOT / 'build_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(report, ensure_ascii=False, indent=2))
print('Report saved to', report_path)


In [ ]:
# Cell 6 — smoke test: load full retriever and run a query
from llm_speaker_core.retrieval.service import HybridRetrievalService

retriever = HybridRetrievalService.load(
    DATA_ROOT / 'index_manifest.json',
    enable_dense=True,
    enable_reranker=True,
)

hits = retriever.search(SMOKE_TEST_QUERY, top_k=3)
print('hits =', len(hits))
for idx, hit in enumerate(hits, start=1):
    print(f"\n[{idx}] score={hit['score']:.4f}")
    print(hit['source'])
    print(hit['text'][:400].replace('\n', ' '))

assert hits, 'Smoke test returned no hits'


In [ ]:
# Cell 7 — pack artifacts for download
import tarfile

archive_items = [
    (DATA_ROOT / 'index_manifest.json', 'data/index_manifest.json'),
    (DATA_ROOT / 'indexes/bm25/index.json', 'data/indexes/bm25/index.json'),
    (DATA_ROOT / 'indexes/faiss/index.json', 'data/indexes/faiss/index.json'),
    (DATA_ROOT / 'indexes/faiss/index.vectors.npy', 'data/indexes/faiss/index.vectors.npy'),
    (DATA_ROOT / 'indexes/faiss/index.faiss', 'data/indexes/faiss/index.faiss'),
    (DATA_ROOT / 'normalized/documents.jsonl', 'data/normalized/documents.jsonl'),
    (DATA_ROOT / 'normalized/chunks.jsonl', 'data/normalized/chunks.jsonl'),
    (OUTPUT_ROOT / 'build_report.json', 'data/build_report.json'),
]

with tarfile.open(ARCHIVE_PATH, 'w:gz') as tar:
    for src, arcname in archive_items:
        assert src.exists(), f'Missing artifact: {src}'
        tar.add(src, arcname=arcname)

print('Archive ready:', ARCHIVE_PATH)
print('Archive size MB =', round(ARCHIVE_PATH.stat().st_size / 1024 / 1024, 2))
print('Artifacts included:')
for src, arcname in archive_items:
    print('-', arcname)


## Что скачать после сборки

Скачайте `rag_index.tar.gz` из `Output` и распакуйте в корень репозитория.

На рабочей машине дальше достаточно:

```bash
git clone ...
git lfs pull
uv sync
# распаковать rag_index.tar.gz поверх data/
ollama serve
uv run llm-voice-stack --retrieval-mode full ...
```

Если rebuild делался после изменений ingestion/retrieval, именно этот архив и становится новым источником истины для `data/index_manifest.json` и `data/indexes/*`.
